<a href="https://colab.research.google.com/github/howard010-pixel/project_test/blob/main/p_0423.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import requests
import pandas as pd
import numpy as np
import yfinance as yf
import time
import logging
import zipfile
import os
# 解決 Pandas 中文對齊問題
pd.set_option('display.unicode.ambiguous_as_wide', True)
pd.set_option('display.unicode.east_asian_width', True)

# 設定顯示最大欄位數（避免中間被省略）
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
from datetime import datetime, timedelta
from pathlib import Path
# 先安裝: pip install tabulate
from tabulate import tabulate


# ============================================================
# 全域設定
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("etf_pipeline.log", encoding="utf-8")
    ]
)
logger = logging.getLogger(__name__)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

# 資料夾建立
Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/clean").mkdir(parents=True, exist_ok=True)

# 追蹤的 ETF 清單
TAIWAN_ETFS = {
    "0050": "元大台灣50",
    "0051": "元大中型100",
    "0052": "富邦科技",
    "0056": "元大高股息",
    "00878": "國泰永續高股息",
    "00881": "國泰台灣5G+",
    "00692": "富邦公司治理",
    "006208": "富邦台50",
}

ETF_CODES = list(TAIWAN_ETFS.keys())


# ============================================================
# 第一部分：爬蟲
# ============================================================

def fetch_institutional_investors(date: str) -> pd.DataFrame | None:
    """
    爬取單日三大法人買賣超資料（來源：台灣證交所）

    參數：
        date (str)：日期字串，格式 'YYYYMMDD'，例如 '20240101'
    回傳：
        DataFrame 或 None（查無資料時）
    """
    url = "https://www.twse.com.tw/rwd/zh/fund/T86"
    params = {
        "response": "json",
        "date": date,
        "selectType": "ALLBUT0999"  # 全部股票（排除 ETN）
    }

    try:
        res = requests.get(url, params=params, headers=HEADERS, timeout=10)
        res.raise_for_status()
        data = res.json()

        if data.get("stat") != "OK":
            logger.warning(f"查無資料（可能為假日或休市）：{date}")
            return None

        df = pd.DataFrame(data["data"], columns=data["fields"])
        df["日期原始"] = date
        logger.info(f"✅ 三大法人 {date}：共 {len(df)} 筆")
        return df

    except requests.exceptions.Timeout:
        logger.error(f"❌ 連線逾時：{date}")
        return None
    except requests.exceptions.ConnectionError:
        logger.error(f"❌ 網路連線失敗：{date}，請檢查網路")
        return None
    except Exception as e:
        logger.error(f"❌ 未知錯誤 {date}：{e}")
        return None


def fetch_institutional_range(start_date: str, end_date: str, delay: float = 3.5) -> pd.DataFrame:
    """
    批次爬取日期區間的三大法人買賣超資料
    自動跳過週末，每次請求間隔 delay 秒（避免被 TWSE 封鎖）

    參數：
        start_date (str)：開始日期 'YYYYMMDD'
        end_date   (str)：結束日期 'YYYYMMDD'
        delay      (float)：請求間隔秒數，建議 3~5 秒
    回傳：
        合併後的 DataFrame
    """
    start = datetime.strptime(start_date, "%Y%m%d")
    end = datetime.strptime(end_date, "%Y%m%d")
    all_data = []
    current = start
    total_days = (end - start).days + 1
    fetched = 0

    logger.info(f"📡 開始爬取三大法人資料：{start_date} → {end_date}")

    while current <= end:
        # 跳過週末（週六=5, 週日=6）
        if current.weekday() < 5:
            date_str = current.strftime("%Y%m%d")
            df = fetch_institutional_investors(date_str)
            if df is not None:
                all_data.append(df)
                fetched += 1
            time.sleep(delay)
        current += timedelta(days=1)

    if not all_data:
        logger.warning("⚠️  無任何法人資料被爬取")
        return pd.DataFrame()

    result = pd.concat(all_data, ignore_index=True)

    # 儲存原始資料
    save_path = f"data/raw/institutional_{start_date}_{end_date}.csv"
    result.to_csv(save_path, index=False, encoding="utf-8-sig")
    logger.info(f"📁 三大法人原始資料已儲存：{save_path}（共 {fetched} 個交易日）")

    return result


def fetch_etf_prices(start_date: str, end_date: str) -> pd.DataFrame:
    """
    使用 yfinance 批次下載所有 ETF 歷史股價
    自動計算技術指標：5/20 日均線、日報酬率、5日累積報酬

    參數：
        start_date (str)：'YYYY-MM-DD'
        end_date   (str)：'YYYY-MM-DD'
    回傳：
        合併所有 ETF 的 DataFrame
    """
    all_price = []

    logger.info(f"📡 開始下載 ETF 股價：{start_date} → {end_date}")

    for code, name in TAIWAN_ETFS.items():
        ticker = f"{code}.TW"
        try:
            stock = yf.Ticker(ticker)
            df = stock.history(start=start_date, end=end_date)

            if df.empty:
                logger.warning(f"⚠️  {name}（{ticker}）無資料")
                continue

            df = df.reset_index()

            # 統一欄位名稱
            df.rename(columns={
                "Date": "日期",
                "Open": "開盤",
                "High": "最高",
                "Low": "最低",
                "Close": "收盤",
                "Volume": "成交量"
            }, inplace=True)

            df["代號"] = code
            df["ETF名稱"] = name

            # 技術指標
            df["5日均價"] = df["收盤"].rolling(5, min_periods=1).mean().round(2)
            df["20日均價"] = df["收盤"].rolling(20, min_periods=1).mean().round(2)
            df["日報酬率"] = df["收盤"].pct_change().round(4)
            df["5日累積報酬"] = df["收盤"].pct_change(5).round(4)

            # 只保留需要的欄位
            df = df[["日期", "代號", "ETF名稱", "開盤", "最高", "最低",
                     "收盤", "成交量", "5日均價", "20日均價", "日報酬率", "5日累積報酬"]]

            all_price.append(df)
            logger.info(f"✅ {name}（{ticker}）：{len(df)} 筆")

        except Exception as e:
            logger.error(f"❌ {ticker} 下載失敗：{e}")

    if not all_price:
        logger.warning("⚠️  無任何 ETF 股價資料")
        return pd.DataFrame()

    result = pd.concat(all_price, ignore_index=True)

    save_path = f"data/raw/etf_prices_{start_date}_{end_date}.csv"
    result.to_csv(save_path, index=False, encoding="utf-8-sig")
    logger.info(f"📁 ETF 股價已儲存：{save_path}")

    return result


# ============================================================
# 第二部分：資料清洗
# ============================================================

def clean_institutional(df: pd.DataFrame) -> pd.DataFrame:
    """
    清洗三大法人原始資料
    步驟：欄位重命名 → 移除千分位逗號 → 轉數值 → 統一日期格式
          → 補齊合計欄 → 填補空值 → 過濾異常值
    """
    if df.empty:
        return df

    df = df.copy()

    # 1. 欄位重命名（相容 TWSE 不同版本欄位名）
    rename_map = {
        "證券代號": "代號",
        "證券名稱": "名稱",
        "外陸資買進股數": "外資買進",
        "外陸資賣出股數": "外資賣出",
        "外陸資買賣超股數": "外資買賣超",
        "投信買進股數": "投信買進",
        "投信賣出股數": "投信賣出",
        "投信買賣超股數": "投信買賣超",
        "自營商買賣超股數": "自營買賣超",
        "三大法人買賣超股數": "三大法人合計",
        "日期原始": "日期",
    }
    df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}, inplace=True)

    # 2. 移除千分位逗號，轉為數值
    numeric_cols = ["外資買進", "外資賣出", "外資買賣超",
                    "投信買進", "投信賣出", "投信買賣超",
                    "自營買賣超", "三大法人合計"]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = (df[col].astype(str)
                       .str.replace(",", "", regex=False)
                       .str.replace(" ", "", regex=False)
                       .str.strip())
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # 3. 日期格式統一（處理民國年 / 西元年）
    if "日期" in df.columns:
        def parse_date(d):
            d = str(d).strip()
            # 民國年格式：113/01/01
            if "/" in d and len(d.split("/")[0]) <= 3:
                parts = d.split("/")
                year = int(parts[0]) + 1911
                return pd.Timestamp(f"{year}-{parts[1]}-{parts[2]}")
            # 西元年數字格式：20240101
            try:
                return pd.to_datetime(d, format="%Y%m%d")
            except Exception:
                return pd.NaT

        df["日期"] = df["日期"].apply(parse_date)

    # 4. 補齊三大法人合計欄（若原始資料缺少）
    if "三大法人合計" not in df.columns:
        df["三大法人合計"] = (
                df.get("外資買賣超", 0).fillna(0) +
                df.get("投信買賣超", 0).fillna(0) +
                df.get("自營買賣超", 0).fillna(0)
        )

    # 5. 填補數值空值為 0
    for col in numeric_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # 6. 過濾異常值（超過 4 個標準差視為資料錯誤）
    col = "三大法人合計"
    if col in df.columns and len(df) > 10:
        mean, std = df[col].mean(), df[col].std()
        before = len(df)
        df = df[abs(df[col] - mean) <= 4 * std].copy()
        removed = before - len(df)
        if removed > 0:
            logger.info(f"🔧 異常值過濾：移除 {removed} 筆（超過 4σ）")

    # 7. 只保留需要的欄位
    keep = ["日期", "代號", "名稱", "外資買賣超", "投信買賣超", "自營買賣超", "三大法人合計"]
    df = df[[c for c in keep if c in df.columns]]

    logger.info(f"✅ 三大法人清洗完成：{len(df)} 筆")
    return df


def filter_etf_only(df: pd.DataFrame) -> pd.DataFrame:
    """從三大法人資料中篩選出 ETF 標的"""
    if "代號" not in df.columns or df.empty:
        return df
    etf_df = df[df["代號"].isin(ETF_CODES)].copy()
    logger.info(f"📌 ETF 篩選結果：{len(etf_df)} 筆（共 {etf_df['代號'].nunique()} 檔 ETF）")
    return etf_df


def add_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    新增滾動統計特徵，提供給預測模型使用：
    - 5/10 日移動平均
    - 5 日累積買賣方向（正數=偏多，負數=偏空）
    - 連續買超/賣超天數
    """
    if df.empty:
        return df

    df = df.sort_values(["代號", "日期"]).copy()

    for col in ["外資買賣超", "投信買賣超", "三大法人合計"]:
        if col not in df.columns:
            continue

        # 移動平均
        df[f"{col}_5日均"] = (
            df.groupby("代號")[col]
            .transform(lambda x: x.rolling(5, min_periods=1).mean())
            .round(0)
        )
        df[f"{col}_10日均"] = (
            df.groupby("代號")[col]
            .transform(lambda x: x.rolling(10, min_periods=1).mean())
            .round(0)
        )

        # 5日累積方向（+1=買超, -1=賣超, 0=持平）
        df[f"{col}_5日方向"] = (
            df.groupby("代號")[col]
            .transform(lambda x: x.rolling(5, min_periods=1)
                       .apply(lambda s: int((s > 0).sum()) - int((s < 0).sum())))
        )

    # 連續買超天數（正數=連買N天, 負數=連賣N天）
    def streak(series):
        result = []
        count = 0
        for val in series:
            if val > 0:
                count = count + 1 if count > 0 else 1
            elif val < 0:
                count = count - 1 if count < 0 else -1
            else:
                count = 0
            result.append(count)
        return result

    if "三大法人合計" in df.columns:
        df["三大法人_連續天數"] = (
            df.groupby("代號")["三大法人合計"]
            .transform(streak)
        )

    logger.info("✅ 滾動特徵新增完成")
    return df


def merge_with_price(institutional_df: pd.DataFrame, price_df: pd.DataFrame) -> pd.DataFrame:
    """
    合併法人買賣超資料與 ETF 股價資料
    以（日期, 代號）為 key 做 left join
    """
    if institutional_df.empty or price_df.empty:
        logger.warning("⚠️  其中一個 DataFrame 為空，無法合併")
        return institutional_df

    # Ensure both '日期' columns are timezone-naive for merging compatibility
    institutional_df["日期"] = pd.to_datetime(institutional_df["日期"]).dt.tz_localize(None)
    price_df["日期"] = pd.to_datetime(price_df["日期"]).dt.tz_localize(None)

    # 只取需要的股價欄位
    price_cols = ["日期", "代號", "收盤", "成交量", "5日均價", "20日均價",
                  "日報酬率", "5日累積報酬"]
    price_sub = price_df[[c for c in price_cols if c in price_df.columns]]

    merged = pd.merge(institutional_df, price_sub, on=["日期", "代號"], how="left")
    logger.info(f"✅ 合併完成：{len(merged)} 筆，股價覆蓋率 {merged['收盤'].notna().mean() * 100:.1f}%")
    return merged


# ============================================================
# 第三部分：資料驗證
# ============================================================

def validate(df: pd.DataFrame, label: str = "資料") -> dict:
    """
    執行資料品質驗證，回傳問題清單
    檢查項目：空值、日期缺口、重複資料、數值合理性、ETF 覆蓋率
    """
    report = {"label": label, "total_rows": len(df), "issues": []}

    if df.empty:
        report["issues"].append("❌ DataFrame 完全為空")
        return report

    # 1. 空值檢查
    null_counts = df.isnull().sum()
    for col, cnt in null_counts[null_counts > 0].items():
        pct = cnt / len(df) * 100
        report["issues"].append(f"⚠️  [{col}] 有 {cnt} 筆空值（{pct:.1f}%）")

    # 2. 日期連續性（找超過 7 天的缺口）
    if "日期" in df.columns:
        dates = pd.to_datetime(df["日期"]).drop_duplicates().sort_values()
        gaps = dates.diff().dropna()
        for i, gap in enumerate(gaps):
            if gap > pd.Timedelta(days=7):
                d1 = dates.iloc[i]
                d2 = dates.iloc[i + 1]
                report["issues"].append(f"⚠️  日期缺口：{d1.date()} → {d2.date()}（{gap.days} 天）")

    # 3. 重複資料
    if "日期" in df.columns and "代號" in df.columns:
        dup = df.duplicated(subset=["日期", "代號"]).sum()
        if dup > 0:
            report["issues"].append(f"❌ 重複資料：{dup} 筆（同日期同代號）")

    # 4. 數值合理性
    for col in ["外資買賣超", "三大法人合計"]:
        if col in df.columns:
            max_val = df[col].abs().max()
            if max_val > 5e8:
                report["issues"].append(f"⚠️  [{col}] 最大值 {max_val:,.0f} 張，請確認是否異常")

    # 5. ETF 覆蓋率
    if "代號" in df.columns:
        for code in ["0050", "0056", "00878"]:
            if (df["代號"] == code).sum() == 0:
                report["issues"].append(f"❌ ETF {code} 完全缺失")

    # 輸出報告
    status = "✅ 全部通過" if not report["issues"] else f"發現 {len(report['issues'])} 個問題"
    logger.info(f"\n{'=' * 55}")
    logger.info(f"  驗證報告：{label}")
    logger.info(f"  資料筆數：{len(df)}　{status}")
    for issue in report["issues"]:
        logger.info(f"  {issue}")
    logger.info(f"{'=' * 55}\n")

    return report


def generate_summary(df: pd.DataFrame) -> pd.DataFrame:
    """產生各 ETF 統計摘要"""
    if "代號" not in df.columns or df.empty:
        return pd.DataFrame()

    agg_dict = {"日期": ["count", "min", "max"]}
    for col in ["外資買賣超", "投信買賣超", "三大法人合計"]:
        if col in df.columns:
            agg_dict[col] = ["mean", "std", "sum"]

    summary = df.groupby("代號").agg(agg_dict)
    summary.columns = ["_".join(c).strip() for c in summary.columns]
    summary = summary.round(0)

    logger.info(f"\n{'=' * 55}")
    logger.info("  各 ETF 資料摘要")
    logger.info(f"\n{summary.to_string()}")
    logger.info(f"{'=' * 55}\n")

    return summary


# ============================================================
# 第四部分：主流程
# ============================================================

def run_pipeline(
        start_date: str = "20240101",
        end_date: str = None,
        request_delay: float = 3.5,
        skip_crawl: bool = False
) -> pd.DataFrame:
    """
    執行完整流程：爬蟲 → 清洗 → 特徵工程 → 驗證 → 輸出

    參數：
        start_date    (str)：開始日期 'YYYYMMDD'
        end_date      (str)：結束日期 'YYYYMMDD'，預設為今天
        request_delay (float)：爬蟲間隔秒數（建議 3~5）
        skip_crawl    (bool)：True 則跳過爬蟲，直接讀取現有 raw 資料
    回傳：
        清洗後的最終 DataFrame
    """
    if end_date is None:
        end_date = datetime.today().strftime("%Y%m%d")

    # 股價用的日期格式
    price_start = f"{start_date[:4]}-{start_date[4:6]}-{start_date[6:]}"
    price_end = f"{end_date[:4]}-{end_date[4:6]}-{end_date[6:]}"

    logger.info(f"\n{'=' * 55}")
    logger.info(f"  🚀 啟動資料流程")
    logger.info(f"  期間：{start_date} → {end_date}")
    logger.info(f"{'=' * 55}\n")

    # ── Step 1：爬蟲 ─────────────────────────────────────
    if skip_crawl:
        logger.info("⏭️  跳過爬蟲，讀取現有原始資料...")
        inst_path = f"data/raw/institutional_{start_date}_{end_date}.csv"
        price_path = f"data/raw/etf_prices_{price_start}_{price_end}.csv"

        if not Path(inst_path).exists():
            logger.error(f"❌ 找不到原始資料：{inst_path}")
            return pd.DataFrame()

        raw_inst = pd.read_csv(inst_path, dtype=str)
        raw_price = pd.read_csv(price_path, dtype={"代號": str}) if Path(price_path).exists() else pd.DataFrame()
    else:
        logger.info("📡 Step 1：爬取三大法人買賣超...")
        raw_inst = fetch_institutional_range(start_date, end_date, request_delay)

        logger.info("📡 Step 1b：下載 ETF 歷史股價...")
        raw_price = fetch_etf_prices(price_start, price_end)

    if raw_inst.empty:
        logger.error("❌ 無法繼續：三大法人資料為空")
        return pd.DataFrame()

    # ── Step 2：清洗 ─────────────────────────────────────
    logger.info("🧹 Step 2：清洗三大法人資料...")
    clean_inst = clean_institutional(raw_inst)

    logger.info("🔍 Step 2b：篩選 ETF 標的...")
    etf_inst = filter_etf_only(clean_inst)

    # ── Step 3：特徵工程 ──────────────────────────────────
    logger.info("⚙️  Step 3：新增滾動特徵...")
    etf_inst = add_rolling_features(etf_inst)

    # ── Step 4：合併股價 ──────────────────────────────────
    if not raw_price.empty:
        logger.info("🔗 Step 4：合併 ETF 股價資料...")
        final_df = merge_with_price(etf_inst, raw_price)
    else:
        logger.warning("⚠️  無股價資料，跳過合併步驟")
        final_df = etf_inst

    # ── Step 5：驗證 ─────────────────────────────────────
    logger.info("✔️  Step 5：執行資料驗證...")
    validate(final_df, "ETF 三大法人最終資料")
    generate_summary(final_df)

    # ── Step 6：儲存 ─────────────────────────────────────
    save_path = "data/clean/etf_institutional_final.csv"
    final_df.to_csv(save_path, index=False, encoding="utf-8-sig")
    logger.info(f"💾 最終資料已儲存：{save_path}（{len(final_df)} 筆）")

    logger.info("\n✅ 全部流程完成！\n")
    return final_df


# ============================================================
# 入口點
# ============================================================

if __name__ == "__main__":
    # ────────────────────────────────────────────
    # ✏️  在這裡修改你要爬取的日期區間
    START_DATE = "20240101"  # 開始日期（YYYYMMDD）
    END_DATE = "20240106"  # 結束日期（YYYYMMDD），或改成 None 表示到今天
    DELAY = 3.5  # 每次請求間隔秒數（請勿低於 3 秒）
    SKIP_CRAWL = False  # True = 跳過爬蟲，直接用已下載的 raw 資料
    # ────────────────────────────────────────────

    df = run_pipeline(
        start_date=START_DATE,
        end_date=END_DATE,
        request_delay=DELAY,
        skip_crawl=SKIP_CRAWL
    )

    if not df.empty:
        print("最終資料預覽（前 10 筆）：")
        # headers="keys" 表示使用 DataFrame 的欄位名，tablefmt="grid" 會畫出漂亮框線
        print(tabulate(df.head(10), headers="keys", tablefmt="grid", showindex=False))

最終資料預覽（前 10 筆）：
+---------------------+--------+-------------+--------------+--------------+----------------+--------------------+---------------------+----------------------+----------------------+-----------------------+------------------------+---------------------+----------+----------+-----------+------------+------------+---------------+
| 日期                |   代號 | 名稱        |   投信買賣超 |   自營買賣超 |   三大法人合計 |   投信買賣超_5日均 |   投信買賣超_10日均 |   投信買賣超_5日方向 |   三大法人合計_5日均 |   三大法人合計_10日均 |   三大法人合計_5日方向 |   三大法人_連續天數 |     收盤 |   成交量 |   5日均價 |   20日均價 |   日報酬率 |   5日累積報酬 |
+=====================+========+=============+==============+==============+================+====================+=====================+======================+======================+=======================+========================+=====================+==========+==========+===========+============+============+===============+
| 2024-01-02 00:00:00 |   0050 | 元大台灣50  |            0 |      -899623 |         575845 |  

In [5]:
"""
============================================================
三大法人買賣超 ETF 資料爬蟲與數據清理系統
============================================================

只需安裝 pandas：
    pip install pandas

執行方式：
    直接在 PyCharm 按 Run ▶️

輸出檔案：
    ETF_Summary.csv        → 爬蟲原始資料
    ETF_Clean.csv          → 清洗後資料
    ETF_Validation.txt     → 驗證報告
============================================================
"""

import urllib.request as req
import pandas as pd
import time
import json
import random
from pathlib import Path

# ============================================================
# 全域設定（可自行修改）
# ============================================================

START_DATE = "2026-04-01"   # 開始日期
END_DATE   = "2026-04-10"   # 結束日期

# ============================================================
# 第一部分：爬蟲
# ============================================================

def get_twse_etf_data(date_str: str) -> pd.DataFrame | None:
    """
    爬取單日三大法人買賣超資料（來源：台灣證交所）
    只保留 ETF 相關欄位
    """
    url = f"https://www.twse.com.tw/rwd/zh/fund/T86?response=json&date={date_str}&selectType=0099P"

    try:
        resp = req.urlopen(url, timeout=10)
        data = json.loads(resp.read().decode("utf-8"))

        if data.get("stat") == "OK":
            df = pd.DataFrame(data["data"], columns=data["fields"])

            # 只保留需要的欄位
            selected_cols = [
                "證券代號",
                "證券名稱",
                "外陸資買賣超股數(不含外資自營商)",
                "投信買賣超股數",
                "自營商買賣超股數",
                "三大法人買賣超股數"
            ]
            df = df[selected_cols]

            # 加入日期欄位
            df.insert(0, "日期", date_str)
            return df

    except Exception as e:
        print(f"日期 {date_str} 抓取異常: {e}")

    return None


def run_crawler(start_date: str, end_date: str) -> pd.DataFrame:
    """
    批次爬取日期區間資料
    自動跳過假日，有重試機制，隨機間隔防封鎖
    """
    # 產生工作日清單（週一～週五）
    date_list = pd.date_range(
        start=start_date, end=end_date, freq="B"
    ).strftime("%Y%m%d")

    all_dfs = []

    print(f"\n{'='*55}")
    print(f"  開始爬取：{start_date} → {end_date}")
    print(f"  共 {len(date_list)} 個工作日")
    print(f"{'='*55}\n")

    for d in date_list:
        success = False

        # 重試機制，最多嘗試 3 次
        for attempt in range(3):
            print(f"正在抓取 {d}（嘗試 {attempt + 1}）...", end=" ")
            res = get_twse_etf_data(d)

            if res is not None:
                all_dfs.append(res)
                print("✅ 成功")
                success = True
                break
            else:
                if attempt < 2:
                    print("❌ 無資料，等待重試...")
                    time.sleep(5)
                else:
                    print("❌ 確定無資料（休市或假日）")

        # 隨機停頓，避免被封鎖
        time.sleep(random.uniform(3, 7))

    # 合併所有資料
    if not all_dfs:
        print("\n⚠️  沒有抓到任何資料")
        return pd.DataFrame()

    result = pd.concat(all_dfs, ignore_index=True)

    # 儲存原始資料
    result.to_csv("ETF_Summary.csv", encoding="utf-8-sig", index=False)
    print(f"\n📁 原始資料已儲存：ETF_Summary.csv（共 {len(result)} 筆）")

    return result


# ============================================================
# 第二部分：數據清理
# ============================================================

def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    清洗三大法人原始資料
    步驟：
        1. 移除千分位逗號，轉為數值
        2. 統一日期格式
        3. 移除重複資料
        4. 填補空值為 0
        5. 過濾異常值（超過 4 個標準差）
        6. 新增計算欄位
    """
    if df.empty:
        print("⚠️  輸入資料為空，跳過清洗")
        return df

    df = df.copy()

    print(f"\n{'='*55}")
    print(f"  開始清洗資料（原始：{len(df)} 筆）")
    print(f"{'='*55}")

    # ── 1. 移除千分位逗號，轉為數值 ─────────────────────
    numeric_cols = [
        "外陸資買賣超股數(不含外資自營商)",
        "投信買賣超股數",
        "自營商買賣超股數",
        "三大法人買賣超股數"
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = (
                df[col].astype(str)
                       .str.replace(",", "", regex=False)
                       .str.replace(" ", "", regex=False)
                       .str.strip()
            )
            df[col] = pd.to_numeric(df[col], errors="coerce")

    print("✅ 步驟1：千分位逗號移除，轉為數值完成")

    # ── 2. 統一日期格式 ──────────────────────────────────
    df["日期"] = pd.to_datetime(df["日期"], format="%Y%m%d", errors="coerce")
    print("✅ 步驟2：日期格式統一完成")

    # ── 3. 移除重複資料 ──────────────────────────────────
    before = len(df)
    df = df.drop_duplicates(subset=["日期", "證券代號"])
    removed = before - len(df)
    if removed > 0:
        print(f"✅ 步驟3：移除重複資料 {removed} 筆")
    else:
        print("✅ 步驟3：無重複資料")

    # ── 4. 填補空值為 0 ──────────────────────────────────
    null_count = df[numeric_cols].isnull().sum().sum()
    df[numeric_cols] = df[numeric_cols].fillna(0)
    print(f"✅ 步驟4：填補空值 {null_count} 筆（補 0）")

    # ── 5. 過濾異常值（超過 4 個標準差）─────────────────
    col = "三大法人買賣超股數"
    if col in df.columns and len(df) > 10:
        mean = df[col].mean()
        std  = df[col].std()
        before = len(df)
        df = df[abs(df[col] - mean) <= 4 * std].copy()
        removed = before - len(df)
        print(f"✅ 步驟5：過濾異常值 {removed} 筆（超過 4σ）")
    else:
        print("✅ 步驟5：資料量不足，跳過異常值過濾")

    # ── 6. 新增計算欄位 ──────────────────────────────────
    df = df.sort_values(["證券代號", "日期"]).reset_index(drop=True)

    # 各法人 5 日移動平均
    for col in numeric_cols:
        if col in df.columns:
            df[f"{col}_5日均"] = (
                df.groupby("證券代號")[col]
                  .transform(lambda x: x.rolling(5, min_periods=1).mean())
                  .round(0)
            )

    # 三大法人連續買超/賣超天數
    def calc_streak(series):
        result = []
        count = 0
        for val in series:
            if val > 0:
                count = count + 1 if count > 0 else 1
            elif val < 0:
                count = count - 1 if count < 0 else -1
            else:
                count = 0
            result.append(count)
        return result

    df["連續買賣超天數"] = (
        df.groupby("證券代號")["三大法人買賣超股數"]
          .transform(calc_streak)
    )

    print("✅ 步驟6：新增 5日均、連續買賣超天數欄位")
    print(f"\n🎉 清洗完成！最終資料：{len(df)} 筆")

    return df


# ============================================================
# 第三部分：資料驗證
# ============================================================

def validate_data(df: pd.DataFrame) -> str:
    """
    驗證清洗後資料品質
    檢查：空值、日期缺口、重複資料、數值合理性、ETF 覆蓋率
    回傳：驗證報告字串
    """
    report_lines = []
    report_lines.append("=" * 55)
    report_lines.append("  資料驗證報告")
    report_lines.append(f"  總筆數：{len(df)}")
    report_lines.append("=" * 55)

    issues = []

    # 1. 空值檢查
    null_counts = df.isnull().sum()
    null_cols   = null_counts[null_counts > 0]
    if null_cols.empty:
        report_lines.append("✅ 空值檢查：無空值")
    else:
        for col, cnt in null_cols.items():
            pct = cnt / len(df) * 100
            msg = f"⚠️  [{col}] 有 {cnt} 筆空值（{pct:.1f}%）"
            report_lines.append(msg)
            issues.append(msg)

    # 2. 日期缺口檢查（超過 7 天視為異常缺口）
    if "日期" in df.columns:
        dates = pd.to_datetime(df["日期"]).drop_duplicates().sort_values()
        gaps  = dates.diff().dropna()
        large_gaps = gaps[gaps > pd.Timedelta(days=7)]
        if large_gaps.empty:
            report_lines.append("✅ 日期檢查：無異常缺口")
        else:
            for i in large_gaps.index:
                pos = dates.tolist().index(dates[i])
                d1  = dates.iloc[pos - 1].date()
                d2  = dates.iloc[pos].date()
                msg = f"⚠️  日期缺口：{d1} → {d2}（可能為連假）"
                report_lines.append(msg)
                issues.append(msg)

    # 3. 重複資料檢查
    if "日期" in df.columns and "證券代號" in df.columns:
        dup = df.duplicated(subset=["日期", "證券代號"]).sum()
        if dup == 0:
            report_lines.append("✅ 重複檢查：無重複資料")
        else:
            msg = f"❌ 發現 {dup} 筆重複資料（同日期同代號）"
            report_lines.append(msg)
            issues.append(msg)

    # 4. 數值合理性
    col = "三大法人買賣超股數"
    if col in df.columns:
        max_val = df[col].abs().max()
        if max_val > 5e8:
            msg = f"⚠️  [{col}] 最大值 {max_val:,.0f}，請確認是否異常"
            report_lines.append(msg)
            issues.append(msg)
        else:
            report_lines.append(f"✅ 數值合理性：最大值 {max_val:,.0f}（正常範圍）")

    # 5. ETF 覆蓋率
    if "證券代號" in df.columns:
        etf_count = df["證券代號"].nunique()
        date_count = df["日期"].nunique() if "日期" in df.columns else "?"
        report_lines.append(f"✅ 覆蓋率：共 {etf_count} 檔 ETF，{date_count} 個交易日")

    # 結論
    report_lines.append("-" * 55)
    if not issues:
        report_lines.append("🎉 結論：資料品質良好，可交給模型組使用")
    else:
        report_lines.append(f"⚠️  結論：發現 {len(issues)} 個問題，請確認後再使用")
    report_lines.append("=" * 55)

    report = "\n".join(report_lines)
    print(f"\n{report}")

    # 儲存驗證報告
    with open("ETF_Validation.txt", "w", encoding="utf-8") as f:
        f.write(report)
    print("\n📁 驗證報告已儲存：ETF_Validation.txt")

    return report


# ============================================================
# 第四部分：統計摘要
# ============================================================

def generate_summary(df: pd.DataFrame):
    """產生各 ETF 統計摘要，方便快速了解資料概況"""
    if "證券代號" not in df.columns or df.empty:
        return

    summary = df.groupby(["證券代號", "證券名稱"]).agg(
        交易日數=("日期", "count"),
        起始日=("日期", "min"),
        結束日=("日期", "max"),
        外資買賣超均值=("外陸資買賣超股數(不含外資自營商)", "mean"),
        投信買賣超均值=("投信買賣超股數", "mean"),
        自營買賣超均值=("自營商買賣超股數", "mean"),
        三大法人均值=("三大法人買賣超股數", "mean"),
    ).round(0)

    print(f"\n{'='*55}")
    print("  各 ETF 統計摘要")
    print(f"{'='*55}")
    print(summary.to_string())
    print(f"{'='*55}\n")


# ============================================================
# 主流程
# ============================================================

if __name__ == "__main__":

    print("""
╔══════════════════════════════════════════════╗
║   三大法人 ETF 爬蟲與數據清理系統             ║
╚══════════════════════════════════════════════╝
    """)

    # ── Step 1：爬蟲 ─────────────────────────────────────
    print("📡 Step 1：開始爬取三大法人資料...\n")
    raw_df = run_crawler(START_DATE, END_DATE)

    if raw_df.empty:
        print("❌ 爬蟲失敗，程式結束")
        exit()

    # ── Step 2：清洗 ─────────────────────────────────────
    print("\n🧹 Step 2：開始清洗資料...")
    clean_df = clean_data(raw_df)

    # ── Step 3：驗證 ─────────────────────────────────────
    print("\n🔍 Step 3：執行資料驗證...")
    validate_data(clean_df)

    # ── Step 4：統計摘要 ──────────────────────────────────
    print("\n📊 Step 4：產生統計摘要...")
    generate_summary(clean_df)

    # ── Step 5：儲存最終清洗資料 ──────────────────────────
    clean_df.to_csv("ETF_Clean.csv", encoding="utf-8-sig", index=False)
    print("📁 清洗資料已儲存：ETF_Clean.csv")

    print("""
╔══════════════════════════════════════════════╗
║   ✅ 全部完成！                               ║
║                                              ║
║   輸出檔案：                                  ║
║   • ETF_Summary.csv   原始爬蟲資料            ║
║   • ETF_Clean.csv     清洗後資料（給模型組）   ║
║   • ETF_Validation.txt 驗證報告              ║
╚══════════════════════════════════════════════╝
    """)


╔══════════════════════════════════════════════╗
║   三大法人 ETF 爬蟲與數據清理系統             ║
╚══════════════════════════════════════════════╝
    
📡 Step 1：開始爬取三大法人資料...


  開始爬取：2026-04-01 → 2026-04-10
  共 8 個工作日

正在抓取 20260401（嘗試 1）... ✅ 成功
正在抓取 20260402（嘗試 1）... ✅ 成功
正在抓取 20260403（嘗試 1）... ❌ 無資料，等待重試...
正在抓取 20260403（嘗試 2）... ❌ 無資料，等待重試...
正在抓取 20260403（嘗試 3）... ❌ 確定無資料（休市或假日）
正在抓取 20260406（嘗試 1）... ❌ 無資料，等待重試...
正在抓取 20260406（嘗試 2）... ❌ 無資料，等待重試...
正在抓取 20260406（嘗試 3）... ❌ 確定無資料（休市或假日）
正在抓取 20260407（嘗試 1）... ✅ 成功
正在抓取 20260408（嘗試 1）... ✅ 成功
正在抓取 20260409（嘗試 1）... ✅ 成功
正在抓取 20260410（嘗試 1）... ✅ 成功

📁 原始資料已儲存：ETF_Summary.csv（共 1297 筆）

🧹 Step 2：開始清洗資料...

  開始清洗資料（原始：1297 筆）
✅ 步驟1：千分位逗號移除，轉為數值完成
✅ 步驟2：日期格式統一完成
✅ 步驟3：無重複資料
✅ 步驟4：填補空值 0 筆（補 0）
✅ 步驟5：過濾異常值 14 筆（超過 4σ）
✅ 步驟6：新增 5日均、連續買賣超天數欄位

🎉 清洗完成！最終資料：1283 筆

🔍 Step 3：執行資料驗證...

  資料驗證報告
  總筆數：1283
✅ 空值檢查：無空值
✅ 日期檢查：無異常缺口
✅ 重複檢查：無重複資料
✅ 數值合理性：最大值 65,305,590（正常範圍）
✅ 覆蓋率：共 218 檔 ETF，6 個交易日
------------------------------------------------------